In [1]:
print(123)

123


In [4]:
%%bash
PREFIX="https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/02-vector-search/embed"
wget "$PREFIX/download.py"
wget "$PREFIX/embedder.py"

--2026-06-29 19:13:39--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/02-vector-search/embed/download.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1376 (1.3K) [text/plain]
Saving to: ‘download.py’

     0K .                                                     100% 34.1M=0s

2026-06-29 19:13:40 (34.1 MB/s) - ‘download.py’ saved [1376/1376]

--2026-06-29 19:13:40--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/02-vector-search/embed/embedder.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.111.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
L

Q1. Embedding a query
Embed the following query:

How does approximate nearest neighbor search work?

The embedder returns a vector of 384 numbers. What's the first value (v[0])?

-0.31

-0.02

-0.12

-0.44

In [2]:
from embedder import Embedder

embed = Embedder()

q = "How does approximate nearest neighbor search work?"
v1 = embed.encode(q)
print(v1[0])

2026-06-29 20:33:53.669457010 [W:onnxruntime:Default, device_discovery.cc:133 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename "5620e0c7-8062-4dce-aeb7-520c7ef76171" did not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+


-0.02058203437252893


Loading the data
We pull the lesson pages from the course repository, the same way as in homework 1. We pin to commit 8c1834d so everyone works with the same data.

In [3]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

Each document is a dictionary with a filename and content, and there are 72 pages.



#### Q2. Cosine similarity
The embedder returns normalized vectors, so the dot product between two of them is their cosine similarity.

Take the page 02-vector-search/lessons/07-sqlitesearch-vector.md, embed its content, and compute the cosine similarity with the query vector from Q1. What do you get?

 0.07

 0.37

 0.68
 
 0.92

In [4]:
import numpy as np

# 1. Grab the content from index 22
content = documents[22]['content']

# 2. Embed this specific page
v2 = embed.encode(content)

# 3. Compute the similarity with your Q1 query vector (v1)
similarity = np.dot(v1, v2)

print(f"Cosine Similarity: {similarity:.4f}")

NameError: name 'v1' is not defined

### Q3. Chunking and search by hand
A full page covers several topics, which waters down its embedding.

We chunk the pages the same way as in homework 1:

from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)
We embed every chunk's content with encode_batch, stack the vectors into a matrix X, and score the Q1 query against all chunks:

scores = X.dot(v)
Which file does the highest-scoring chunk belong to (its filename)?

02-vector-search/lessons/03-embeddings-dataset.md
02-vector-search/lessons/06-rag-vector.md
02-vector-search/lessons/07-sqlitesearch-vector.md
02-vector-search/lessons/09-onnx-embedder.md

v1.dot(dv)


In [34]:
import numpy as np
from gitsource import chunk_documents

In [33]:
# 1. Chunk your entire documents collection exactly as described
chunks = chunk_documents(documents, size=2000, step=1000)
print(f"Total chunks created: {len(chunks)}")

Total chunks created: 295


In [35]:
chunk_texts = [chunk['content'] for chunk in chunks]

In [ ]:
X = embed.encode_batch(chunk_texts)

In [ ]:
import numpy as np
from gitsource import chunk_documents

# 1. Chunk your entire documents collection exactly as described
chunks = chunk_documents(documents, size=2000, step=1000)
print(f"Total chunks created: {len(chunks)}")

# 2. Extract the text content from each chunk into a clean list
# (Handles cases where chunks are dictionaries or raw objects)
chunk_texts = [chunk['content'] if isinstance(chunk, dict) else chunk for chunk in chunks]

# 3. Embed all text chunks using encode_batch to form your matrix X
# This returns a 2D NumPy array of shape (num_chunks, embedding_dim)
X = embed.encode_batch(chunk_texts)

# 4. Prepare your Q1 query vector (v)
query_text = "How does approximate nearest neighbor search work?"
v = embed.encode(query_text)

# 5. Calculate similarity scores using the matrix dot product
scores = X.dot(v)

# 6. Locate the index of the absolute highest score
highest_score_idx = np.argmax(scores)
print(f"Highest similarity score achieved: {scores[highest_score_idx]:.4f}")

# 7. Map the winning chunk back to its source filename metadata
winning_chunk = chunks[highest_score_idx]

if isinstance(winning_chunk, dict) and 'filename' in winning_chunk:
    print(f"🏆 Winning File: {winning_chunk['filename']}")
else:
    # Fallback: If chunks lost their metadata keys, search for the text match in the original docs
    winning_text = chunk_texts[highest_score_idx]
    for doc in documents:
        if winning_text in doc['content']:
            print(f"🏆 Winning File (via text matching): {doc.get('filename', 'Unknown')}")
            break